In [3]:
import pandas as pd
import numpy as np

In [4]:
score = pd.read_csv('../../data/processed/week2_score.csv', index_col=0, parse_dates=True)
rank = pd.read_csv('../../data/processed/week2_sector_rank.csv', index_col=0, parse_dates=True)
drawdown = pd.read_csv('../../data/processed/week2_drawdown.csv', index_col=0, parse_dates=True)
volatility = pd.read_csv('../../data/processed/week2_volatility.csv', index_col=0, parse_dates=True)

In [5]:
print("score shape:", score.shape)
print("rank shape:", rank.shape)
print("drawdown shape:", drawdown.shape)
print("volatility shape:", volatility.shape)

score shape: (1644, 11)
rank shape: (1644, 11)
drawdown shape: (1644, 12)
volatility shape: (1644, 12)


In [6]:
drawdown = drawdown.drop(columns=['SPY'])
volatility = volatility.drop(columns=['SPY'])

In [7]:
print("score shape:", score.shape)
print("rank shape:", rank.shape)
print("drawdown shape:", drawdown.shape)
print("volatility shape:", volatility.shape)

score shape: (1644, 11)
rank shape: (1644, 11)
drawdown shape: (1644, 11)
volatility shape: (1644, 11)


In [9]:
monthly_score = score.resample('ME').last()
monthly_rank = rank.resample('ME').last()
monthly_drawdown = drawdown.resample('ME').last()
monthly_volatility = volatility.resample('ME').last()

In [10]:
monthly_score.head()

,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2018-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-08-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-09-30,-0.051087,-0.068947,-0.019116,-0.010043,0.072322,0.078009,0.027906,-0.029718,-0.015898,0.139445,0.068083
2018-10-31,-0.243726,-0.172023,-0.254729,-0.155617,-0.210201,-0.141536,0.014615,-0.075643,-0.009470,-0.084025,-0.165920


In [11]:
drawdown_filter = monthly_drawdown >= -0.15
drawdown_filter.head()


,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2018-06-30,True,True,True,True,True,True,True,True,True,True,True
2018-07-31,True,True,True,True,True,True,True,True,True,True,True
2018-08-31,True,True,True,True,True,True,True,True,True,True,True
2018-09-30,True,True,True,True,True,True,True,True,True,True,True
2018-10-31,True,True,True,True,True,True,True,True,True,True,True


In [12]:
vol_median = monthly_volatility.median(axis=1)
vol_median.head()

Date
2018-06-30         NaN
2018-07-31         NaN
2018-08-31         NaN
2018-09-30    0.007586
2018-10-31    0.010557
Freq: ME, dtype: float64

In [13]:
volatility_filter = monthly_volatility.le(vol_median, axis=0)
volatility_filter.head()

,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2018-06-30,False,False,False,False,False,False,False,False,False,False,False
2018-07-31,False,False,False,False,False,False,False,False,False,False,False
2018-08-31,False,False,False,False,False,False,False,False,False,False,False
2018-09-30,False,False,False,False,True,True,True,True,False,True,True
2018-10-31,False,False,False,True,True,False,True,True,True,True,False


In [14]:
combined_filter = drawdown_filter & volatility_filter
combined_filter.head()


,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2018-06-30,False,False,False,False,False,False,False,False,False,False,False
2018-07-31,False,False,False,False,False,False,False,False,False,False,False
2018-08-31,False,False,False,False,False,False,False,False,False,False,False
2018-09-30,False,False,False,False,True,True,True,True,False,True,True
2018-10-31,False,False,False,True,True,False,True,True,True,True,False


In [15]:
filtered_score = monthly_score.where(combined_filter)
filtered_score.head()

,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
Date,,,,,,,,,,,
2018-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-08-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-09-30,NaN,NaN,NaN,NaN,0.072322,0.078009,0.027906,-0.029718,NaN,0.139445,0.068083
2018-10-31,NaN,NaN,NaN,-0.155617,-0.210201,NaN,0.014615,-0.075643,-0.00947,-0.084025,NaN


In [16]:
top_sectors = filtered_score.apply(
    lambda row: row.nlargest(3).dropna().index.tolist(),
    axis=1
)

top_sectors.head()

Date
2018-06-30                  []
2018-07-31                  []
2018-08-31                  []
2018-09-30     [XLV, XLK, XLI]
2018-10-31    [XLP, XLU, XLRE]
Freq: ME, dtype: object

In [17]:
allocations = top_sectors.apply(
    lambda sectors: {s: 1/3 for s in sectors} if len(sectors) == 3 else {}
)

allocations.head()

Date
2018-06-30                                                   {}
2018-07-31                                                   {}
2018-08-31                                                   {}
2018-09-30    {'XLV': 0.3333333333333333, 'XLK': 0.333333333...
2018-10-31    {'XLP': 0.3333333333333333, 'XLU': 0.333333333...
Freq: ME, dtype: object